In [1]:
# Importing the necessary libraries
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import ExtraTreesRegressor, AdaBoostRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor, BaggingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import (HuberRegressor, LassoCV, ElasticNetCV, LassoLarsCV,
                                   LassoLarsIC, LarsCV, Lars, RANSACRegressor, 
                                   ElasticNet, Lasso, OrthogonalMatchingPursuitCV, 
                                   OrthogonalMatchingPursuit, LassoLars,
                                   RidgeCV, BayesianRidge, Ridge)
from sklearn.compose import TransformedTargetRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.dummy import DummyRegressor

In [2]:
# Helpers for loading the processed main dataset and imputing missing weather features from the training split only
def resolve_main_dataset_path():
    candidate_paths = [
        Path("..") / "floodprediction" / "Flood-Prevention-Using-River-Level-Prediction" / "datasets" / "processed" / "weather_rainfall_riverlevel_20212025" / "v20260309_02" / "weatherrainfallrivermaindataset.parquet",
        Path("datasets") / "processed" / "weather_rainfall_riverlevel_20212025" / "v20260309_02" / "weatherrainfallrivermaindataset.parquet",
    ]

    for candidate_path in candidate_paths:
        if candidate_path.exists():
            return candidate_path

    searched_paths = "\n".join(str(path.resolve()) for path in candidate_paths)
    raise FileNotFoundError(
        "Could not find weatherrainfallrivermaindataset.parquet. Searched:\n"
        f"{searched_paths}"
    )

def load_and_preprocess_data(file_path, selected_columns=None):
    data = pd.read_parquet(file_path)
    data = data.rename(columns={"timestamp": "Timestamp", "location": "river_location"})

    if selected_columns is not None:
        missing_columns = [column for column in selected_columns if column not in data.columns]
        if missing_columns:
            raise KeyError(f"Missing expected columns: {missing_columns}")
        data = data.loc[:, selected_columns].copy()

    data["Timestamp"] = pd.to_datetime(data["Timestamp"])
    data = data.sort_values("Timestamp").reset_index(drop=True)
    return data

def impute_training_means(X_train, X_test, columns_to_impute):
    """Fill missing values using means learned from the training split only."""
    imputation_values = {}

    for column in columns_to_impute:
        train_mean = X_train[column].mean()
        if pd.isna(train_mean):
            train_mean = 0.0
            print(f"Warning: {column} is fully null in training data; using 0.0")

        train_nulls = X_train[column].isna().sum()
        test_nulls = X_test[column].isna().sum()

        if train_nulls or test_nulls:
            print(
                f"Imputing {column}: train nulls={train_nulls}, "
                f"test nulls={test_nulls}, mean={train_mean:.4f}"
            )

        X_train.loc[:, column] = X_train[column].fillna(train_mean)
        X_test.loc[:, column] = X_test[column].fillna(train_mean)
        imputation_values[column] = train_mean

    return X_train, X_test, imputation_values

In [10]:
# Drop rows with missing targets before creating feature and response arrays
def split_features_target(data, target_column):
    supervised_data = data.dropna(subset=[target_column]).copy()
    dropped_rows = len(data) - len(supervised_data)

    if dropped_rows:
        print(f"Dropping {dropped_rows} rows with missing {target_column} before modeling")

    X = supervised_data.drop(target_column, axis=1)
    y = supervised_data[target_column]
    return X, y

In [4]:
# Chronological split keeps the past in training and the future in testing
def split_train_test_chronological(X, y, test_size=0.2):
    n_rows = len(X)
    split_idx = int(n_rows * (1 - test_size))
    X_train, X_test = X.iloc[:split_idx].copy(), X.iloc[split_idx:].copy()
    y_train, y_test = y.iloc[:split_idx].copy(), y.iloc[split_idx:].copy()
    return X_train, X_test, y_train, y_test

In [5]:
# Scale features using statistics learned from the training split only
def scale_features(X_train, X_test):
    feature_scaler = StandardScaler()
    X_train_scaled = pd.DataFrame(
        feature_scaler.fit_transform(X_train),
        columns=X_train.columns,
        index=X_train.index,
    )
    X_test_scaled = pd.DataFrame(
        feature_scaler.transform(X_test),
        columns=X_test.columns,
        index=X_test.index,
    )

    return X_train_scaled, X_test_scaled, feature_scaler

In [6]:
parquet_path = resolve_main_dataset_path()
feature_columns = [
    'humidity',
    'temperature',
    'windspeed',
    'dewpoint',
    'pressure',
    'rainfall_reading',
    'wind_direction',
]
target_column = 'river_level'
selected_columns = ['Timestamp', 'river_location', *feature_columns, target_column]

full_df = load_and_preprocess_data(parquet_path, selected_columns)
full_df['Year'] = full_df['Timestamp'].dt.year
available_locations = sorted(full_df['river_location'].dropna().unique())
available_years = sorted(full_df['Year'].dropna().unique())

print(f"Dataset path: {parquet_path.resolve()}")
print(f"Available locations: {len(available_locations)}")
print(full_df['river_location'].value_counts().sort_index())
print()
print(f"Available years: {available_years}")
print(f"Total rows: {len(full_df):,}")
print(f"Date range: {full_df['Timestamp'].min()} -> {full_df['Timestamp'].max()}")

Dataset path: C:\PhD\floodprediction\Flood-Prevention-Using-River-Level-Prediction\datasets\processed\weather_rainfall_riverlevel_20212025\v20260309_02\weatherrainfallrivermaindataset.parquet
Available locations: 4
river_location
Ballymun Coultry Parks Depot    2439090
Civic Offices                   2439090
Kippure                         3360780
Pigeon House                    2439090
Name: count, dtype: int64

Available years: [np.int32(2021), np.int32(2022), np.int32(2023), np.int32(2024), np.int32(2025)]
Total rows: 10,678,050
Date range: 2021-01-01 00:00:00 -> 2025-12-31 23:59:00


In [7]:
# Defining a dictionary containing the different models
models = {
    "Extra Trees Regressor": ExtraTreesRegressor(random_state=420),
    "AdaBoost Regressor": AdaBoostRegressor(random_state=420),
    "Gradient Boosting Regressor": GradientBoostingRegressor(random_state=420),
    "HistGradient Boosting Regressor": HistGradientBoostingRegressor(random_state=420),
    "K Neighbors Regressor": KNeighborsRegressor(),
    "Bagging Regressor": BaggingRegressor(random_state=420),
    "Huber Regressor": HuberRegressor(),
    "Transformed Target Regressor": TransformedTargetRegressor(regressor=HuberRegressor(), transformer=StandardScaler()),
    "LassoCV": LassoCV(),
    "ElasticNetCV": ElasticNetCV(),
    "LassoLarsCV": LassoLarsCV(),
    "LassoLarsIC": LassoLarsIC(),
    "LarsCV": LarsCV(),
    "Lars": Lars(),
    "RANSAC Regressor": RANSACRegressor(random_state=420),
    "ElasticNet": ElasticNet(),
    "Lasso": Lasso(),
    "Orthogonal Matching Pursuit CV": OrthogonalMatchingPursuitCV(),
    "Orthogonal Matching Pursuit": OrthogonalMatchingPursuit(),
    "Ridge": Ridge(),
    "RidgeCV": RidgeCV(),
    "BayesianRidge": BayesianRidge(),
    "Dummy Regressor": DummyRegressor(),
    "Lasso Lars": LassoLars(),
}

In [8]:
def evaluate_model(model, X_train, y_train, X_test, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    r2 = r2_score(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    
    return rmse, mse, mae, r2

In [11]:
import time
from copy import deepcopy

all_location_year_results = {}
min_rows_required = 10

for location in available_locations:
    location_years = sorted(
        full_df.loc[full_df['river_location'] == location, 'Year'].dropna().unique()
    )

    for year in location_years:
        print(f"\n{'='*60}")
        print(f"  LOCATION: {location} | YEAR: {year}")
        print(f"{'='*60}")

        location_year_df = (
            full_df[
                (full_df['river_location'] == location)
                & (full_df['Year'] == year)
            ]
            .sort_values('Timestamp')
            .reset_index(drop=True)
        )
        print(f"Rows before target filtering: {len(location_year_df)}")

        modeling_df = location_year_df.drop(['Timestamp', 'river_location', 'Year'], axis=1)
        X, y = split_features_target(modeling_df, target_column)
        print(f"Usable supervised rows: {len(X)}")

        if len(X) < min_rows_required:
            print(f"Skipping {location} {year}: not enough non-null target rows for train/test evaluation")
            continue

        X_train, X_test, y_train, y_test = split_train_test_chronological(X, y)
        print(f"Train: {len(X_train)} rows (first 80%) | Test: {len(X_test)} rows (last 20%)")

        if len(X_train) == 0 or len(X_test) == 0:
            print(f"Skipping {location} {year}: empty train or test split")
            continue

        X_train, X_test, imputation_values = impute_training_means(X_train, X_test, feature_columns)
        print(f"Imputation means: {imputation_values}")

        X_train_scaled, X_test_scaled, _ = scale_features(X_train, X_test)

        results = {}
        for index, (name, model) in enumerate(models.items(), 1):
            print(f"  [{index}/{len(models)}] {name}...", end=" ", flush=True)
            start = time.time()
            model_copy = deepcopy(model)
            rmse, mse, mae, r2 = evaluate_model(
                model_copy, X_train_scaled, y_train, X_test_scaled, y_test
            )
            elapsed = time.time() - start
            results[name] = {"RMSE": rmse, "MSE": mse, "MAE": mae, "R2": r2}
            print(f"done in {elapsed:.1f}s | R2={r2:.4f}")

        location_year_results_df = pd.DataFrame(results).T.sort_values('R2', ascending=False)
        all_location_year_results[(location, year)] = location_year_results_df
        print(f"\n--- {location} | {year} Results (sorted by R2) ---")
        print(location_year_results_df)

print(f"\n{'='*60}")
print("  SUMMARY: Best model per location-year")
print(f"{'='*60}")
for (location, year), results_df in all_location_year_results.items():
    best_model = results_df.index[0]
    print(f"  {location} | {year}: {best_model} (R2={results_df.iloc[0]['R2']:.4f})")


  LOCATION: Ballymun Coultry Parks Depot | YEAR: 2021
Rows before target filtering: 478620
Usable supervised rows: 478620
Train: 382896 rows (first 80%) | Test: 95724 rows (last 20%)
Imputing pressure: train nulls=20274, test nulls=1481, mean=1006.4938
Imputing rainfall_reading: train nulls=13, test nulls=0, mean=0.0010
Imputing wind_direction: train nulls=7399, test nulls=0, mean=190.4670
Imputation means: {'humidity': np.float64(77.62538892179937), 'temperature': np.float64(10.758641563494363), 'windspeed': np.float64(1.3647809880115176), 'dewpoint': np.float64(7.8041384284524655), 'pressure': np.float64(1006.493780024378), 'rainfall_reading': np.float64(0.001036870271074976), 'wind_direction': np.float64(190.4669786150888)}
  [1/24] Extra Trees Regressor... done in 96.9s | R2=-3.1776
  [2/24] AdaBoost Regressor... done in 26.1s | R2=-3.1368
  [3/24] Gradient Boosting Regressor... done in 46.5s | R2=-3.0996
  [4/24] HistGradient Boosting Regressor... done in 2.3s | R2=-3.1069
  [5/2

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit

rf_results = {}
min_rows_required = 10
min_train_rows_for_cv = 8

for location in available_locations:
    location_years = sorted(
        full_df.loc[full_df['river_location'] == location, 'Year'].dropna().unique()
    )

    for year in location_years:
        print(f"\n{'='*60}")
        print(f"  RandomForestRegressor LOCATION: {location} | YEAR: {year}")
        print(f"{'='*60}")

        location_year_df = (
            full_df[
                (full_df['river_location'] == location)
                & (full_df['Year'] == year)
            ]
            .sort_values('Timestamp')
            .reset_index(drop=True)
        )
        print(f"Rows before target filtering: {len(location_year_df)}")

        modeling_df = location_year_df.drop(['Timestamp', 'river_location', 'Year'], axis=1)
        X, y = split_features_target(modeling_df, target_column)
        print(f"Usable supervised rows: {len(X)}")

        if len(X) < min_rows_required:
            print(f"Skipping {location} {year}: not enough non-null target rows for train/test evaluation")
            continue

        X_train, X_test, y_train, y_test = split_train_test_chronological(X, y)
        print(f"Train: {len(X_train)} rows (first 80%) | Test: {len(X_test)} rows (last 20%)")

        if len(X_train) < min_train_rows_for_cv or len(X_test) == 0:
            print(f"Skipping {location} {year}: not enough rows for TimeSeriesSplit cross-validation")
            continue

        X_train, X_test, imputation_values = impute_training_means(X_train, X_test, feature_columns)
        print(f"Imputation means: {imputation_values}")

        X_train_scaled, X_test_scaled, _ = scale_features(X_train, X_test)

        param_grid = {
            'n_estimators': [50, 100],
            'max_depth': [12, 18],
            'min_samples_split': [15, 20],
            'min_samples_leaf': [12, 18],
            'max_features': ['sqrt', 'log2', None],
        }
        rf_regressor = RandomForestRegressor(random_state=420)
        grid_search = GridSearchCV(
            estimator=rf_regressor,
            param_grid=param_grid,
            cv=TimeSeriesSplit(n_splits=3),
            n_jobs=-1,
            scoring='neg_mean_squared_error',
            verbose=1,
        )
        grid_search.fit(X_train_scaled, y_train)

        best_rf = grid_search.best_estimator_
        y_pred = best_rf.predict(X_test_scaled)
        mse = mean_squared_error(y_test, y_pred)
        mae = mean_absolute_error(y_test, y_pred)
        rmse = np.sqrt(mse)
        r2 = r2_score(y_test, y_pred)

        print(f"Best params: {grid_search.best_params_}")
        print(f"Test MSE: {mse:.6f} | MAE: {mae:.6f} | RMSE: {rmse:.6f} | R2: {r2:.4f}")
        rf_results[(location, year)] = {
            'params': grid_search.best_params_,
            'MSE': mse,
            'MAE': mae,
            'RMSE': rmse,
            'R2': r2,
        }

print("\n=== RandomForestRegressor Summary ===")
for (location, year), result in rf_results.items():
    print(
        f"{location} | {year}: R2={result['R2']:.4f}, RMSE={result['RMSE']:.6f}, "
        f"Best params={result['params']}"
    )


  RandomForestRegressor LOCATION: Ballymun Coultry Parks Depot | YEAR: 2021
Rows before target filtering: 478620
Usable supervised rows: 478620
Train: 382896 rows (first 80%) | Test: 95724 rows (last 20%)
Imputing pressure: train nulls=20274, test nulls=1481, mean=1006.4938
Imputing rainfall_reading: train nulls=13, test nulls=0, mean=0.0010
Imputing wind_direction: train nulls=7399, test nulls=0, mean=190.4670
Imputation means: {'humidity': np.float64(77.62538892179937), 'temperature': np.float64(10.758641563494363), 'windspeed': np.float64(1.3647809880115176), 'dewpoint': np.float64(7.8041384284524655), 'pressure': np.float64(1006.493780024378), 'rainfall_reading': np.float64(0.001036870271074976), 'wind_direction': np.float64(190.4669786150888)}
Fitting 3 folds for each of 48 candidates, totalling 144 fits
Best params: {'max_depth': 12, 'max_features': 'sqrt', 'min_samples_leaf': 12, 'min_samples_split': 15, 'n_estimators': 100}
Test MSE: 0.027709 | MAE: 0.150039 | RMSE: 0.166460 |

c:\PhD\floodpaper\.venv\Lib\site-packages\joblib\externals\loky\process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Best params: {'max_depth': 12, 'max_features': 'sqrt', 'min_samples_leaf': 18, 'min_samples_split': 15, 'n_estimators': 50}
Test MSE: 0.646740 | MAE: 0.682531 | RMSE: 0.804201 | R2: -0.0829

  RandomForestRegressor LOCATION: Kippure | YEAR: 2021
Rows before target filtering: 478620
Usable supervised rows: 478620
Train: 382896 rows (first 80%) | Test: 95724 rows (last 20%)
Imputing pressure: train nulls=6886, test nulls=2, mean=976.4028
Imputing rainfall_reading: train nulls=13, test nulls=0, mean=0.0029
Imputing wind_direction: train nulls=16, test nulls=7, mean=196.3648
Imputation means: {'humidity': np.float64(77.6629065576893), 'temperature': np.float64(8.882040997569968), 'windspeed': np.float64(2.170913863391652), 'dewpoint': np.float64(5.472006859216235), 'pressure': np.float64(976.4027961729741), 'rainfall_reading': np.float64(0.0028651050059678804), 'wind_direction': np.float64(196.3647664995215)}
Fitting 3 folds for each of 48 candidates, totalling 144 fits
Best params: {'max_

KeyboardInterrupt: 

: 